# 🌍 Seismic Data — 2D & Globe Visualization (Final Working Version)

### Why no `Scene()` for custom data?
`Scene.content.add()` only accepts live REST-backed layers with a `.properties` endpoint.  
It **cannot** accept in-memory data (FeatureSet, GeoJSON, drawn points).  
`Map.content.draw()` works perfectly and gives an identical visual result.

### What works ✅
| Feature | Method |
|---|---|
| Live PAGER earthquake feed | `Map` + `gis.content.add(item)` |
| Historical REST layers | `Map` + `safe_load_layer()` |
| Custom colored dots from GeoJSON | `Map` + `content.draw()` |
| Globe-style view | `Map` with `satellite` or `dark-gray-vector` basemap |
| DataFrame query & stats | Living Atlas `FeatureLayer.query()` |

---

## 📦 Step 1 — Install & Import

In [ ]:
!pip install arcgis arcgis-mapping --quiet
print('✅ Ready')

In [ ]:
import json, urllib.request
import pandas as pd
import arcgis
from arcgis.gis         import GIS
from arcgis.geocoding   import geocode
from arcgis.geometry    import Geometry
from arcgis.layers      import MapImageLayer
from arcgis.map.symbols import SimpleMarkerSymbolEsriSMS

gis = GIS()
print('✅ Connected:', gis)

# ── Constants ─────────────────────────────────────────────────────────────────
LIVING_ATLAS_ID = '9e2f2b544c954fda9cd13b7f3e6eebce'
BASE            = 'https://earthquake.usgs.gov/arcgis/rest/services'
HISTORICAL_URL  = f'{BASE}/eq/EQ_Catalog/MapServer'
PLATES_URL      = f'{BASE}/eq/map_plateboundaries/MapServer'
FAULTS_URL      = f'{BASE}/eq/map_faults/MapServer'

GEOJSON = {
    'significant_month': 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/significant_month.geojson',
    'M4_5_month'       : 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_month.geojson',
    'M2_5_month'       : 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_month.geojson',
    'all_day'          : 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson',
}
print('✅ Constants set')

## 🔧 Step 2 — Helper Functions

In [ ]:
# ── Color helpers ─────────────────────────────────────────────────────────────
def mag_to_color(mag):
    if   mag >= 7.0: return [180, 0,   0,   255]
    elif mag >= 6.0: return [255, 50,  0,   230]
    elif mag >= 5.0: return [255, 140, 0,   220]
    elif mag >= 4.0: return [255, 220, 0,   200]
    else:            return [100, 200, 100, 180]

def depth_to_color(depth):
    if   depth <= 30 : return [255, 50,  50,  230]
    elif depth <= 100: return [255, 165, 0,   220]
    elif depth <= 300: return [100, 100, 255, 210]
    else:              return [50,  0,   180, 200]

def mag_to_size(mag):
    return max(6, int((mag - 2) * 5))

def make_symbol(color, size):
    """Build pydantic symbol model — required by Map.content.draw()."""
    return SimpleMarkerSymbolEsriSMS(**{
        'type'   : 'esriSMS',
        'style'  : 'esriSMSCircle',
        'color'  : color,
        'size'   : size,
        'angle'  : 0, 'xoffset': 0, 'yoffset': 0,
        'outline': {'type': 'esriSLS', 'style': 'esriSLSSolid',
                    'color': [0, 0, 0, 80], 'width': 0.5}
    })

# ── Map factory ───────────────────────────────────────────────────────────────
def make_map(location='World', basemap='dark-gray-vector', zoom=2, height='600px'):
    """Create a map widget with sensible defaults."""
    m = gis.map(location)
    m.basemap.basemap  = basemap
    m.zoom             = zoom
    m.layout.height    = height
    return m

# ── Safe REST loader ──────────────────────────────────────────────────────────
def safe_load_layer(url, name):
    """Load a MapImageLayer — returns None silently if the server is down."""
    try:
        lyr = MapImageLayer(url)
        _   = lyr.properties
        print(f'  ✅ {name}')
        return lyr
    except Exception:
        print(f'  ❌ {name} — server unavailable, skipping')
        return None

# ── GeoJSON fetcher ───────────────────────────────────────────────────────────
def fetch_earthquakes(feed='M4_5_month'):
    """Fetch USGS GeoJSON feed → clean DataFrame."""
    print(f'📡 Fetching {feed}...')
    with urllib.request.urlopen(GEOJSON[feed]) as r:
        data = json.loads(r.read().decode())
    rows = []
    for f in data['features']:
        p          = f['properties']
        lon, lat, depth = f['geometry']['coordinates']
        rows.append({
            'mag'  : p.get('mag'),
            'place': p.get('place'),
            'time' : pd.to_datetime(p.get('time'), unit='ms'),
            'depth': depth,
            'lon'  : lon,
            'lat'  : lat,
        })
    df = (pd.DataFrame(rows)
            .dropna(subset=['mag','lat','lon'])
            .sort_values('mag', ascending=False)
            .reset_index(drop=True))
    print(f'✅ {len(df)} earthquakes  |  max mag: {df["mag"].max()}  |  avg depth: {df["depth"].mean():.1f} km')
    return df

# ── Draw engine ───────────────────────────────────────────────────────────────
def draw_earthquakes(map_widget, df, color_by='magnitude'):
    """
    Draw earthquake points on a Map widget using Map.content.draw().
    color_by: 'magnitude' or 'depth'
    NOTE: Only works on Map (gis.map()), NOT on Scene().
    """
    for _, row in df.iterrows():
        pt    = Geometry({'x': row['lon'], 'y': row['lat'],
                          'spatialReference': {'wkid': 4326}})
        color = mag_to_color(row['mag']) if color_by == 'magnitude' \
                else depth_to_color(row['depth'])
        map_widget.content.draw(pt, symbol=make_symbol(color, mag_to_size(row['mag'])))
    print(f'✅ Drew {len(df)} earthquakes  (color_by={color_by})')

print('✅ All helpers ready')

---
## 🗺️ Step 3 — Live Earthquake Map (ArcGIS Living Atlas)
Updated every 5 minutes. Built-in symbology: dot size = magnitude, color = PAGER alert level.

In [ ]:
live_map = make_map(zoom=2)

item = gis.content.get(LIVING_ATLAS_ID)
print(f'📦 {item.title}  |  {item.type}')
print(f'   {item.snippet}')

live_map.content.add(item)
print('\n✅ Live feed added — click any dot for magnitude, depth, PAGER alert')
live_map

---
## 📚 Step 4 — Historical Catalog + Fault Lines + Plate Boundaries

In [ ]:
hist_map = make_map(zoom=2)

print('Loading USGS REST layers...')
plates = safe_load_layer(PLATES_URL,    'Tectonic Plate Boundaries')
faults = safe_load_layer(FAULTS_URL,    'Global Fault Lines')
hist   = safe_load_layer(HISTORICAL_URL,'Historical Seismicity 1900–2024 (M5.5+)')

for lyr in [plates, faults, hist]:
    if lyr: hist_map.content.add(lyr)

hist_map.content.add(gis.content.get(LIVING_ATLAS_ID))
print('  ✅ Living Atlas recent feed')
hist_map

---
## 📍 Step 5 — Focus on Any Region
Change `REGION` to zoom into any seismically active area.

In [ ]:
# ✏️ Options: 'Japan' | 'Indonesia' | 'Chile' | 'Turkey' | 'California, USA' | 'India' | 'Alaska'
REGION = 'Japan'

region_map = make_map(location=REGION, zoom=6)

for lyr in [safe_load_layer(HISTORICAL_URL,'Historical'),
            safe_load_layer(FAULTS_URL,    'Faults')]:
    if lyr: region_map.content.add(lyr)

region_map.content.add(gis.content.get(LIVING_ATLAS_ID))
print(f'\n✅ Focused on: {REGION}')
region_map

---
## 🔢 Step 6 — Query Earthquake Data as a DataFrame

In [ ]:
eq_layer = gis.content.get(LIVING_ATLAS_ID).layers[0]
print('Layer:', eq_layer)

fset = eq_layer.query(where='1=1', out_fields='*', return_geometry=True)
df   = fset.sdf
print(f'\n✅ {len(df)} records  |  Columns: {df.columns.tolist()}')
df.head(10)

In [ ]:
# ── Filter — confirmed column names from actual output ────────────────────────
# mag | depth | place | eventTime | latitude | longitude | alert | sig | magType
big = df[df['mag'] >= 5.0].sort_values('mag', ascending=False)
print(f'M5.0+ events: {len(big)}')
big[['mag', 'depth', 'place', 'eventTime', 'alert']].head(15)

In [ ]:
print('📊 Seismic Statistics:')
print(f'   Total events  : {len(df)}')
print(f'   Max magnitude : {df["mag"].max()}')
print(f'   Avg magnitude : {df["mag"].mean():.2f}')
print(f'   M6.0+ events  : {len(df[df["mag"] >= 6.0])}')
print(f'   Avg depth (km): {df["depth"].mean():.1f}')
print(f'   Deepest (km)  : {df["depth"].max()}')
print('\n🚨 PAGER Alert breakdown:')
print(df['alert'].value_counts().to_string())

---
## 🌍 Step 7 — Fetch Live GeoJSON Data (M4.5+ last 30 days)

In [ ]:
# ✏️ Options: 'significant_month' | 'M4_5_month' | 'M2_5_month' | 'all_day'
geo_df = fetch_earthquakes('M4_5_month')
geo_df.head(10)

---
## 🔴 Step 8 — Globe Map: Colored by MAGNITUDE
Dark satellite basemap + magnitude-colored dots from GeoJSON.

In [ ]:
mag_map = make_map(basemap='satellite', zoom=2, height='650px')

draw_earthquakes(mag_map, geo_df, color_by='magnitude')

print('\nColor  = magnitude  |  Size = magnitude')
print('  🔴 Dark red  = M7.0+  |  🔴 Red    = M6.0–6.9')
print('  🟠 Orange    = M5.0–5.9  |  🟡 Yellow = M4.0–4.9')
mag_map

---
## 🔵 Step 9 — Globe Map: Colored by DEPTH
Same data, color now shows earthquake depth — reveals **subduction zones** as deep blue clusters.

In [ ]:
depth_map = make_map(basemap='dark-gray-vector', zoom=2, height='650px')

draw_earthquakes(depth_map, geo_df, color_by='depth')

print('\nColor = depth  |  Size = magnitude')
print('  🔴 Red        = Shallow   0–30 km   (crust/surface faults)')
print('  🟠 Orange     = Mid       30–100 km (upper mantle)')
print('  🔵 Blue       = Deep      100–300 km (subduction zones)')
print('  🟣 Dark blue  = Very deep 300+ km   (deep slab)')
depth_map

---
## 🗾 Step 10 — Regional Focus Map with Custom Dots

In [ ]:
# ✏️ Change to any region
FOCUS = 'Japan'

res = geocode(FOCUS, max_locations=1)

focus_map = make_map(location=FOCUS, basemap='satellite', zoom=5, height='650px')

if res:
    clat = res[0]['location']['y']
    clon = res[0]['location']['x']
    # Filter to ±20° bounding box around region
    region_df = geo_df[
        (geo_df['lat'].between(clat - 20, clat + 20)) &
        (geo_df['lon'].between(clon - 20, clon + 20))
    ].copy()
    print(f'📍 Region: {FOCUS}  |  Earthquakes in bounding box: {len(region_df)}')
    draw_earthquakes(focus_map, region_df, color_by='depth')
else:
    print('Could not geocode region — drawing all earthquakes')
    draw_earthquakes(focus_map, geo_df, color_by='depth')

focus_map

---
## 📊 Step 11 — SeDF Plot (Magnitude Gradient)
Uses the ArcGIS `spatial.plot()` renderer on the Living Atlas DataFrame.

In [ ]:
plot_map = make_map(zoom=2)

# 'mag' confirmed as correct column from actual output
df.spatial.plot(
    map_widget    = plot_map,
    renderer_type = 'c',
    method        = 'esriClassifyNaturalBreaks',
    class_count   = 5,
    col           = 'mag',
    cmap          = 'Reds',
    alpha         = 0.8
)

print('✅ Plotted — darker red = stronger magnitude')
plot_map

---
## 📋 Cheat Sheet

### Core pattern
```python
m = make_map('Japan', basemap='satellite', zoom=6)
geo_df = fetch_earthquakes('M4_5_month')
draw_earthquakes(m, geo_df, color_by='depth')   # or 'magnitude'
m
```

### Why Scene() is NOT used
```
Scene.content.draw()  → does not exist (SceneContent has no draw method)
Scene.content.add(FeatureSet)  → crashes: 'FeatureSet has no attribute properties'
Scene.content.add(MapImageLayer)  → crashes: 'NoneType has no attribute pop'
Scene.content.add(gis.content.get(...))  → crashes: 'NoneType does not support item assignment'
✅ Solution: gis.map() + satellite basemap = identical visual, fully working
```

### Confirmed column names (Living Atlas)
`mag` · `depth` · `place` · `eventTime` · `latitude` · `longitude` · `alert` · `sig` · `magType`

### GeoJSON feeds (always live)
| Key | Contents |
|---|---|
| `significant_month` | Significant EQ past 30 days |
| `M4_5_month` | All M4.5+ past 30 days |
| `M2_5_month` | All M2.5+ past 30 days |
| `all_day` | All EQ today |

### Regions to explore
🇯🇵 Japan · 🇮🇩 Indonesia · 🇨🇱 Chile · 🇹🇷 Turkey · 🇺🇸 California · 🇮🇳 India · 🌊 Ring of Fire